# L17 - La stessa rete in Keras e in PyTorch

**Questo notebook va eseguito in Google Colab**, dove Keras (con TensorFlow) e PyTorch sono già installati. Non funziona in JupyterLite e non è previsto per WinPython: le due librerie occupano diversi GB.

Il problema è quello della lezione L16: le due lune, con una rete di 8 neuroni nascosti con attivazione sigmoide e un neurone di uscita sigmoide. Eseguire le celle in ordine.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def lune(n=200, rumore=0.15, seme=0):
    rng = np.random.default_rng(seme)
    m = n // 2
    t = rng.uniform(0, np.pi, m)
    a = np.column_stack([np.cos(t), np.sin(t)])
    b = np.column_stack([1 - np.cos(t), 0.5 - np.sin(t)])
    X = np.vstack([a, b]) + rng.normal(0, rumore, size=(2 * m, 2))
    y = np.concatenate([np.zeros(m), np.ones(m)])
    return X.astype("float32"), y.astype("float32")

X, y = lune()
print(X.shape, y.shape)

## 1. Keras

Keras descrive la rete come una sequenza di strati:

- `keras.Input(shape=(2,))`: ogni esempio ha 2 ingressi
- `keras.layers.Dense(8, activation="sigmoid")`: strato completamente connesso (dense) con 8 neuroni; contiene la matrice dei pesi e il vettore dei bias, cioè `W1` e `b1` della lezione L16
- `compile`: sceglie l'ottimizzatore (qui la discesa del gradiente, `SGD`), la funzione di perdita e le metriche da mostrare
- `fit`: addestra; `epochs` è il numero di epoche, `batch_size` il numero di esempi usati per ogni aggiornamento dei pesi

In [ ]:
import keras

keras.utils.set_random_seed(0)
modello_k = keras.Sequential([
    keras.Input(shape=(2,)),
    keras.layers.Dense(8, activation="sigmoid"),
    keras.layers.Dense(1, activation="sigmoid"),
])
modello_k.compile(optimizer=keras.optimizers.SGD(learning_rate=2.0),
                  loss="mean_squared_error",
                  metrics=["accuracy"])
modello_k.summary()

In [ ]:
storia = modello_k.fit(X, y, epochs=2000, batch_size=len(X), verbose=0)
print("perdita finale:", storia.history["loss"][-1])
print("accuratezza finale:", storia.history["accuracy"][-1])
plt.plot(storia.history["loss"])
plt.yscale("log")
plt.xlabel("epoca")
plt.ylabel("perdita")
plt.title("Keras")
plt.show()

In [ ]:
W1, b1 = modello_k.layers[0].get_weights()     # pesi e bias del primo strato
print(W1.shape, b1.shape)
previsioni = modello_k.predict(X[:5], verbose=0)
print(np.round(previsioni.ravel(), 3), y[:5])

## 2. PyTorch

In PyTorch il ciclo di addestramento è scritto esplicitamente, come nella lezione L16; la differenza è che il gradiente viene calcolato automaticamente.

- `torch.tensor`: array di PyTorch (tensore), simile a un array NumPy
- `nn.Linear(2, 8)`: strato con 2 ingressi e 8 neuroni (pesi e bias)
- `nn.Sigmoid()`: funzione di attivazione
- `perdita.backward()`: calcola il gradiente della perdita rispetto a tutti i parametri, con la retropropagazione
- `ottimizzatore.step()`: aggiorna i parametri; `zero_grad()` azzera i gradienti del passo precedente

In [ ]:
import torch
from torch import nn

torch.manual_seed(0)
Xt = torch.tensor(X)
yt = torch.tensor(y).reshape(-1, 1)

modello_t = nn.Sequential(
    nn.Linear(2, 8),
    nn.Sigmoid(),
    nn.Linear(8, 1),
    nn.Sigmoid(),
)
funzione_perdita = nn.MSELoss()
ottimizzatore = torch.optim.SGD(modello_t.parameters(), lr=2.0)

perdite = []
for epoca in range(2000):
    y_prev = modello_t(Xt)                   # calcolo in avanti
    perdita = funzione_perdita(y_prev, yt)   # perdita
    ottimizzatore.zero_grad()                # azzera i gradienti precedenti
    perdita.backward()                       # retropropagazione
    ottimizzatore.step()                     # aggiornamento dei parametri
    perdite.append(perdita.item())

accuratezza = ((modello_t(Xt) >= 0.5).float() == yt).float().mean().item()
print("perdita finale:", perdite[-1], " accuratezza:", accuratezza)
plt.plot(perdite)
plt.yscale("log")
plt.xlabel("epoca")
plt.ylabel("perdita")
plt.title("PyTorch")
plt.show()

## 3. Corrispondenze

| Lezione L16 (NumPy) | Keras | PyTorch |
|---|---|---|
| `W1`, `b1` | `Dense(8, ...)` | `nn.Linear(2, 8)` |
| `sigmoide(...)` | `activation="sigmoid"` | `nn.Sigmoid()` |
| `avanti(X, ...)` | `modello.predict(X)` | `modello(X)` |
| perdita MSE | `loss="mean_squared_error"` | `nn.MSELoss()` |
| calcolo di `d2`, `d1`, gradienti | automatico in `fit` | `perdita.backward()` |
| `W = W - eta * g_W` | `SGD(learning_rate=...)` | `ottimizzatore.step()` |
| ciclo sulle epoche | `fit(..., epochs=...)` | ciclo `for` scritto a mano |

## 4. Da provare

- cambiare il numero di neuroni nascosti (2, 4, 32) e osservare accuratezza e curva della perdita
- sostituire `"sigmoid"` con `"relu"` nello strato nascosto (in PyTorch `nn.ReLU()`) e ridurre il tasso di apprendimento a 0.1
- sostituire l'ottimizzatore con `keras.optimizers.Adam(learning_rate=0.01)` o `torch.optim.Adam(modello_t.parameters(), lr=0.01)`
- aggiungere un secondo strato nascosto